# 04 - Train/evaluate top 3 repo models with MLflow
Usa los tres mejores candidatos del repositorio: FinBERT, DistilBERT financiero y DistilRoBERTa financiero. Registra métricas, artefactos y versiones en Unity Catalog Model Registry.

In [0]:
# En Free Edition se usa serverless. Evita dependencias pesadas innecesarias.
# Si el entorno ya trae estas librerías, esta celda no cambia nada.
%pip install -q mlflow scikit-learn pandas cloudpickle transformers torch accelerate typing-extensions --upgrade

In [0]:
dbutils.library.restartPython()

In [0]:
import os
import json
import shutil
import tempfile
from datetime import datetime

import numpy as np
import pandas as pd
import mlflow
import mlflow.pyfunc
from mlflow.models import infer_signature
from mlflow.tracking import MlflowClient

from sklearn.metrics import accuracy_score, f1_score, precision_score, recall_score, classification_report, confusion_matrix

In [0]:
try:
    dbutils.widgets.text("catalog", "workspace")
    dbutils.widgets.text("schema", "financial_sentiment")
    dbutils.widgets.text("experiment_name", "/Shared/financial_sentiment_mlops")
    dbutils.widgets.text("registered_model_basename", "financial_sentiment_champion")
    dbutils.widgets.text("max_eval_rows_per_class", "75")
    dbutils.widgets.text("max_train_rows_per_class", "75")
    dbutils.widgets.text("finetune_epochs", "1")
    dbutils.widgets.text("training_mode", "finetune_small")  # pretrained_eval | finetune_small
    dbutils.widgets.text("allow_sklearn_fallback", "true")
except Exception:
    pass

catalog = dbutils.widgets.get("catalog")
schema = dbutils.widgets.get("schema")
experiment_name = dbutils.widgets.get("experiment_name")
registered_model_basename = dbutils.widgets.get("registered_model_basename")
max_eval_rows_per_class = int(dbutils.widgets.get("max_eval_rows_per_class"))
max_train_rows_per_class = int(dbutils.widgets.get("max_train_rows_per_class"))
finetune_epochs = float(dbutils.widgets.get("finetune_epochs"))
training_mode = dbutils.widgets.get("training_mode")
allow_sklearn_fallback = dbutils.widgets.get("allow_sklearn_fallback").lower() == "true"

full_schema = f"{catalog}.{schema}"
gold_table = f"{full_schema}.gold_financial_sentiment_training"
metrics_table = f"{full_schema}.model_candidate_metrics"
run_log_table = f"{full_schema}.pipeline_run_log"
registered_model_name = f"{full_schema}.{registered_model_basename}"
training_batch_id = f"train_{datetime.utcnow().strftime('%Y%m%d%H%M%S')}"

mlflow.set_experiment(experiment_name)
mlflow.set_registry_uri("databricks-uc")
client = MlflowClient()

print(f"Gold table: {gold_table}")
print(f"Registered model: {registered_model_name}")
print(f"Training batch: {training_batch_id}")

In [0]:
MODEL_SPECS = [
    {
        "model_key": "finbert",
        "model_family": "transformer",
        "hf_model_name": "ProsusAI/finbert",
        "label_order": ["positive", "negative", "neutral"],
        "repo_notebook": "06_finbert_financial_sentiment.ipynb",
    },
    {
        "model_key": "distilbert_financial",
        "model_family": "transformer",
        "hf_model_name": "AnkitAI/distilbert-base-uncased-financial-news-sentiment-analysis",
        "label_order": ["negative", "neutral", "positive"],
        "repo_notebook": "08_distilbert_financial_sentiment.ipynb",
    },
    {
        "model_key": "distilroberta_financial",
        "model_family": "transformer",
        "hf_model_name": "mrm8488/distilroberta-finetuned-financial-news-sentiment-analysis",
        "label_order": ["negative", "neutral", "positive"],
        "repo_notebook": "07_distilroberta_financial_sentiment.ipynb",
    },
]

In [0]:
df = spark.table(gold_table).toPandas()
df = df[["text_clean", "label_normalized", "split"]].dropna().copy()

train_df = df[df["split"] == "train"].copy()
eval_df = df[df["split"].isin(["validation", "test"])].copy()

if len(eval_df) == 0:
    eval_df = df.copy()

# Muestreo pequeño para Free Edition. Mantiene balance por clase y evita consumo excesivo.
eval_df = (
    eval_df
    .groupby("label_normalized", group_keys=False)
    .apply(lambda x: x.sample(n=min(len(x), max_eval_rows_per_class), random_state=42))
    .reset_index(drop=True)
)

print("Train rows:", len(train_df))
print("Eval rows:", len(eval_df))
print(eval_df["label_normalized"].value_counts())

In [0]:
class HuggingFaceSentimentPyFunc(mlflow.pyfunc.PythonModel):
    def load_context(self, context):
        import os
        import torch
        from transformers import AutoTokenizer, AutoModelForSequenceClassification

        self.model_dir = context.artifacts["hf_model"]
        self.tokenizer = AutoTokenizer.from_pretrained(self.model_dir)
        self.model = AutoModelForSequenceClassification.from_pretrained(self.model_dir)
        self.model.eval()

        import json
        with open(os.path.join(self.model_dir, "mlflow_label_order.json"), "r") as f:
            self.label_order = json.load(f)

    def predict(self, context, model_input):
        import pandas as pd
        import torch

        if isinstance(model_input, pd.DataFrame):
            texts = model_input["text_clean"].astype(str).tolist()
        else:
            texts = pd.DataFrame(model_input)["text_clean"].astype(str).tolist()

        predictions, scores = [], []
        batch_size = 16
        max_length = 128

        for i in range(0, len(texts), batch_size):
            batch_texts = texts[i:i + batch_size]
            encoded = self.tokenizer(batch_texts, padding=True, truncation=True, max_length=max_length, return_tensors="pt")
            with torch.no_grad():
                outputs = self.model(**encoded)
                probs = torch.softmax(outputs.logits, dim=1)
                pred_ids = torch.argmax(probs, dim=1).cpu().numpy()
                batch_scores = torch.max(probs, dim=1).values.cpu().numpy()

            predictions.extend([self.label_order[int(idx)] for idx in pred_ids])
            scores.extend(batch_scores.tolist())

        return pd.DataFrame({"prediction": predictions, "score": scores})

class SklearnSentimentPyFunc(mlflow.pyfunc.PythonModel):
    def load_context(self, context):
        import joblib
        self.pipeline = joblib.load(context.artifacts["pipeline"])

    def predict(self, context, model_input):
        import pandas as pd
        if isinstance(model_input, pd.DataFrame):
            texts = model_input["text_clean"].astype(str)
        else:
            texts = pd.DataFrame(model_input)["text_clean"].astype(str)
        preds = self.pipeline.predict(texts)
        if hasattr(self.pipeline, "predict_proba"):
            probs = self.pipeline.predict_proba(texts)
            scores = probs.max(axis=1)
        else:
            scores = [None] * len(preds)
        return pd.DataFrame({"prediction": preds, "score": scores})

In [0]:
def evaluate_predictions(y_true, y_pred):
    return {
        "accuracy": float(accuracy_score(y_true, y_pred)),
        "macro_f1": float(f1_score(y_true, y_pred, average="macro", zero_division=0)),
        "weighted_f1": float(f1_score(y_true, y_pred, average="weighted", zero_division=0)),
        "precision_macro": float(precision_score(y_true, y_pred, average="macro", zero_division=0)),
        "recall_macro": float(recall_score(y_true, y_pred, average="macro", zero_division=0)),
    }

def register_run_model(run_id: str):
    model_uri = f"runs:/{run_id}/model"
    mv = mlflow.register_model(model_uri=model_uri, name=registered_model_name)
    return str(mv.version)

def log_candidate_row(spec, run_id, version, metrics, status="SUCCESS", error_message=None, effective_mode=None):
    row = [(
        training_batch_id,
        spec["model_key"],
        spec.get("model_family", "unknown"),
        spec.get("hf_model_name"),
        effective_mode or training_mode,
        run_id,
        registered_model_name,
        version,
        metrics.get("accuracy") if metrics else None,
        metrics.get("macro_f1") if metrics else None,
        metrics.get("weighted_f1") if metrics else None,
        metrics.get("precision_macro") if metrics else None,
        metrics.get("recall_macro") if metrics else None,
        int(len(train_df)),
        int(len(eval_df)),
        status,
        error_message,
        datetime.utcnow(),
    )]
    schema = "training_batch_id string, model_key string, model_family string, hf_model_name string, training_mode string, run_id string, registered_model_name string, model_version string, accuracy double, macro_f1 double, weighted_f1 double, precision_macro double, recall_macro double, train_rows long, eval_rows long, status string, error_message string, event_ts timestamp"
    spark.createDataFrame(row, schema).write.mode("append").saveAsTable(metrics_table)

In [0]:
def run_transformer_candidate(spec):
    import torch
    from transformers import AutoTokenizer, AutoModelForSequenceClassification

    model_key = spec["model_key"]
    hf_model_name = spec["hf_model_name"]
    label_order = spec["label_order"]

    with tempfile.TemporaryDirectory() as tmpdir:
        model_dir = os.path.join(tmpdir, model_key)
        tokenizer = AutoTokenizer.from_pretrained(hf_model_name)
        model = AutoModelForSequenceClassification.from_pretrained(hf_model_name)

        # Fine-tuning pequeño para Free Edition. Mantiene el entrenamiento real, pero con muestra controlada.
        if training_mode == "finetune_small":
            import torch
            from transformers import Trainer, TrainingArguments

            label_to_id = {label: idx for idx, label in enumerate(label_order)}
            id_to_label = {idx: label for label, idx in label_to_id.items()}
            model.config.label2id = label_to_id
            model.config.id2label = id_to_label

            train_sample = (
                train_df
                .dropna(subset=["text_clean", "label_normalized"])
                .groupby("label_normalized", group_keys=False)
                .apply(lambda x: x.sample(n=min(len(x), max_train_rows_per_class), random_state=42))
                .reset_index(drop=True)
            )

            class SentimentDataset(torch.utils.data.Dataset):
                def __init__(self, texts, labels):
                    self.encodings = tokenizer(
                        texts,
                        truncation=True,
                        padding=True,
                        max_length=128,
                    )
                    self.labels = [label_to_id[str(label)] for label in labels]

                def __getitem__(self, idx):
                    item = {key: torch.tensor(val[idx]) for key, val in self.encodings.items()}
                    item["labels"] = torch.tensor(self.labels[idx])
                    return item

                def __len__(self):
                    return len(self.labels)

            train_dataset = SentimentDataset(
                train_sample["text_clean"].astype(str).tolist(),
                train_sample["label_normalized"].astype(str).tolist(),
            )

            args = TrainingArguments(
                output_dir=os.path.join(tmpdir, "trainer_output"),
                num_train_epochs=finetune_epochs,
                per_device_train_batch_size=8,
                learning_rate=2e-5,
                weight_decay=0.01,
                logging_steps=25,
                save_strategy="no",
                report_to=[],
            )
            trainer = Trainer(model=model, args=args, train_dataset=train_dataset)
            trainer.train()

        model.eval()

        texts = eval_df["text_clean"].astype(str).tolist()
        y_true = eval_df["label_normalized"].astype(str).tolist()
        predictions, scores = [], []

        for i in range(0, len(texts), 16):
            batch = texts[i:i + 16]
            encoded = tokenizer(batch, padding=True, truncation=True, max_length=128, return_tensors="pt")
            with torch.no_grad():
                outputs = model(**encoded)
                probs = torch.softmax(outputs.logits, dim=1)
                pred_ids = torch.argmax(probs, dim=1).cpu().numpy()
                batch_scores = torch.max(probs, dim=1).values.cpu().numpy()
            predictions.extend([label_order[int(idx)] for idx in pred_ids])
            scores.extend(batch_scores.tolist())

        metrics = evaluate_predictions(y_true, predictions)
        report = pd.DataFrame(classification_report(y_true, predictions, output_dict=True, zero_division=0)).transpose()
        confusion = pd.DataFrame(
            confusion_matrix(y_true, predictions, labels=["negative", "neutral", "positive"]),
            index=["real_negative", "real_neutral", "real_positive"],
            columns=["pred_negative", "pred_neutral", "pred_positive"]
        )
        sample_predictions = eval_df.copy()
        sample_predictions["prediction"] = predictions
        sample_predictions["score"] = scores

        os.makedirs(model_dir, exist_ok=True)
        tokenizer.save_pretrained(model_dir)
        model.save_pretrained(model_dir)
        with open(os.path.join(model_dir, "mlflow_label_order.json"), "w") as f:
            json.dump(label_order, f)

        report_path = os.path.join(tmpdir, f"{model_key}_classification_report.csv")
        confusion_path = os.path.join(tmpdir, f"{model_key}_confusion_matrix.csv")
        predictions_path = os.path.join(tmpdir, f"{model_key}_sample_predictions.csv")
        report.to_csv(report_path)
        confusion.to_csv(confusion_path)
        sample_predictions.to_csv(predictions_path, index=False)

        input_example = eval_df[["text_clean"]].head(5)
        output_example = pd.DataFrame({"prediction": predictions[:5], "score": scores[:5]})
        signature = infer_signature(input_example, output_example)

        with mlflow.start_run(run_name=f"candidate_{model_key}_{training_batch_id}") as run:
            mlflow.log_param("project", "ProyectoIntegrador2")
            mlflow.log_param("model_key", model_key)
            mlflow.log_param("hf_model_name", hf_model_name)
            mlflow.log_param("repo_notebook", spec.get("repo_notebook"))
            mlflow.log_param("training_mode", training_mode)
            mlflow.log_param("input_table", gold_table)
            mlflow.log_param("eval_rows", len(eval_df))
            mlflow.log_param("max_train_rows_per_class", max_train_rows_per_class)
            mlflow.log_param("finetune_epochs", finetune_epochs)
            mlflow.log_params({"max_length": 128, "batch_size": 16})
            for k, v in metrics.items():
                mlflow.log_metric(k, v)
            mlflow.log_artifact(report_path)
            mlflow.log_artifact(confusion_path)
            mlflow.log_artifact(predictions_path)
            mlflow.pyfunc.log_model(
                artifact_path="model",
                python_model=HuggingFaceSentimentPyFunc(),
                artifacts={"hf_model": model_dir},
                signature=signature,
                input_example=input_example,
                pip_requirements=["mlflow", "transformers", "torch", "pandas", "cloudpickle"],
            )
            version = register_run_model(run.info.run_id)

        return run.info.run_id, version, metrics

In [0]:
def run_sklearn_fallback(spec):
    from sklearn.pipeline import Pipeline
    from sklearn.feature_extraction.text import TfidfVectorizer
    from sklearn.linear_model import LogisticRegression
    import joblib

    model_key = spec["model_key"] + "_fallback_tfidf"
    y_train = train_df["label_normalized"].astype(str)
    X_train = train_df["text_clean"].astype(str)
    X_eval = eval_df["text_clean"].astype(str)
    y_true = eval_df["label_normalized"].astype(str)

    pipeline = Pipeline([
        ("tfidf", TfidfVectorizer(max_features=10000, ngram_range=(1, 2), min_df=1)),
        ("clf", LogisticRegression(max_iter=500, class_weight="balanced")),
    ])
    pipeline.fit(X_train, y_train)
    y_pred = pipeline.predict(X_eval)
    scores = pipeline.predict_proba(X_eval).max(axis=1) if hasattr(pipeline, "predict_proba") else [None] * len(y_pred)
    metrics = evaluate_predictions(y_true, y_pred)

    fallback_spec = dict(spec)
    fallback_spec["model_key"] = model_key
    fallback_spec["model_family"] = "sklearn_fallback"
    fallback_spec["hf_model_name"] = spec["hf_model_name"]

    with tempfile.TemporaryDirectory() as tmpdir:
        pipeline_path = os.path.join(tmpdir, "pipeline.joblib")
        joblib.dump(pipeline, pipeline_path)

        report = pd.DataFrame(classification_report(y_true, y_pred, output_dict=True, zero_division=0)).transpose()
        report_path = os.path.join(tmpdir, f"{model_key}_classification_report.csv")
        report.to_csv(report_path)

        input_example = eval_df[["text_clean"]].head(5)
        output_example = pd.DataFrame({"prediction": y_pred[:5], "score": scores[:5]})
        signature = infer_signature(input_example, output_example)

        with mlflow.start_run(run_name=f"fallback_{model_key}_{training_batch_id}") as run:
            mlflow.log_param("project", "ProyectoIntegrador2")
            mlflow.log_param("model_key", model_key)
            mlflow.log_param("fallback_for_hf_model", spec["hf_model_name"])
            mlflow.log_param("training_mode", "sklearn_fallback")
            mlflow.log_param("input_table", gold_table)
            for k, v in metrics.items():
                mlflow.log_metric(k, v)
            mlflow.log_artifact(report_path)
            mlflow.pyfunc.log_model(
                artifact_path="model",
                python_model=SklearnSentimentPyFunc(),
                artifacts={"pipeline": pipeline_path},
                signature=signature,
                input_example=input_example,
                pip_requirements=["mlflow", "scikit-learn", "pandas", "cloudpickle", "joblib"],
            )
            version = register_run_model(run.info.run_id)

    return fallback_spec, run.info.run_id, version, metrics

In [0]:
successful = []
for spec in MODEL_SPECS:
    print("=" * 100)
    print(f"Procesando candidato: {spec['model_key']} - {spec['hf_model_name']}")
    try:
        run_id, version, metrics = run_transformer_candidate(spec)
        log_candidate_row(spec, run_id, version, metrics, status="SUCCESS")
        successful.append((spec["model_key"], version, metrics["macro_f1"]))
        print(f"OK: {spec['model_key']} version={version} macro_f1={metrics['macro_f1']:.4f}")
    except Exception as e:
        error = str(e)[:1000]
        print(f"Falló candidato HF {spec['model_key']}: {error}")
        log_candidate_row(spec, None, None, None, status="FAILED", error_message=error)
        if allow_sklearn_fallback:
            print("Ejecutando fallback sklearn para no romper la automatización del pipeline.")
            fallback_spec, run_id, version, metrics = run_sklearn_fallback(spec)
            log_candidate_row(fallback_spec, run_id, version, metrics, status="SUCCESS", effective_mode="sklearn_fallback")
            successful.append((fallback_spec["model_key"], version, metrics["macro_f1"]))
            print(f"Fallback OK: {fallback_spec['model_key']} version={version} macro_f1={metrics['macro_f1']:.4f}")

if len(successful) == 0:
    raise ValueError("Ningún candidato pudo entrenarse/evaluarse ni registrarse.")

spark.createDataFrame([
    (training_batch_id, "05_train_top3_models_mlflow", "SUCCESS", f"Candidatos exitosos: {len(successful)}", int(len(successful)), datetime.utcnow())
], "run_id string, step string, status string, message string, rows_written long, event_ts timestamp").write.mode("append").saveAsTable(run_log_table)

try:
    dbutils.jobs.taskValues.set(key="training_batch_id", value=training_batch_id)
    dbutils.jobs.taskValues.set(key="registered_model_name", value=registered_model_name)
except Exception:
    pass

print("Entrenamiento/evaluación finalizado.")
print(successful)